# Lemma Analysis: Handling Projections and Accessors

## Problem

Some lemmas in tactics are actually **projections or accessors** of larger lemmas, and we're breaking them down incorrectly.

### Example

```
rw [norm_div, mem_sphere_zero_iff_norm.1 x.coe_prop, mem_sphere_zero_iff_norm.1 y.coe_prop, div_one]
```

In this case:
- `mem_sphere_zero_iff_norm.1` is a projection/accessor (the `.1` part)
- The full lemma name is `mem_sphere_zero_iff_norm`
- We should check if the **full name** (including the projection) matches first
- Only if the full name doesn't match, then we should break it down

## Current Behavior

Currently, we might be:
1. Extracting `mem_sphere_zero_iff_norm.1` as a candidate
2. Breaking it down to `mem_sphere_zero_iff_norm` and `1`
3. Trying to match `mem_sphere_zero_iff_norm` (which is correct)
4. But we might miss cases where `mem_sphere_zero_iff_norm.1` itself exists as a full name

## Proposed Solution

1. **First, try to match the full candidate name** (e.g., `mem_sphere_zero_iff_norm.1`)
   - Check if this exact name exists in corpus
   - If found, use it

2. **If not found, then break it down**
   - Split on `.` to get parts
   - Try matching the base name (e.g., `mem_sphere_zero_iff_norm`)
   - Handle projections/accessors appropriately

3. **Consider the context**
   - Projections like `.1`, `.2` are typically tuple/record accessors
   - These should be resolved to the parent lemma name
   - But we should verify the parent exists first

## Implementation Notes

- Update `normalize_candidate()` to check full name first
- Update `resolve_candidate()` to handle projection patterns
- Consider Lean naming conventions:
  - `.1`, `.2`, etc. = tuple/record projections
  - `.mp`, `.mpr` = modus ponens variants
  - Other suffixes might be specific accessors

## References

- Example from tactics: `mem_sphere_zero_iff_norm.1 x.coe_prop`
- Need to check if `mem_sphere_zero_iff_norm.1` exists as a full name
- If not, resolve to `mem_sphere_zero_iff_norm` and verify it exists

# getting lemmas


In [ ]:
# Load theorem registry (for theorem matching)
import json
theorem_registry = []
with open("theorem_registry_all.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            theorem_registry.append(json.loads(line))

print(f"Loaded {len(theorem_registry)} theorems from registry")

# Reload modules first (so we can import load_corpus_premises)
import importlib.util
import sys

module_name = 'myutils2'
file_path = '00_myutils2.py'
if module_name in sys.modules:
    del sys.modules[module_name]

spec = importlib.util.spec_from_file_location(module_name, file_path)
module = importlib.util.module_from_spec(spec)
sys.modules[module_name] = module
spec.loader.exec_module(module)

from myutils2 import *

print("✓ Module reloaded successfully!")

# Load corpus and extract premises (for premise/lemma resolution)
# Use corpus.jsonl instead of premise_registry_unique.jsonl
premise_registry = load_corpus_premises("corpus.jsonl")

print(f"Loaded {len(premise_registry)} premises from corpus")

Loaded 126797 theorems from registry
✓ Module reloaded successfully!
Loaded 180973 premises from corpus


In [ ]:
# Build a dictionary: {thm_name: [candidates]}, and also a total list of all candidates (all occurrences, no thm info)
from tqdm import tqdm

thm_to_candidates = {}
all_candidates_occurrences = []

# Add a progress bar for all data points
for i in tqdm(range(len(data)), desc="Extracting candidates", unit="proof"):
    all_candidates = extract_all_premises(
        i=i,
        data=data,
        printing=False
    )
    if all_candidates is None:
        # Stopped printing for each
        continue

    # Theorem name assumed as data[i][0]
    thm = data[i][0]
    cands_in_proof = [candidate for _, candidate in all_candidates]
    thm_to_candidates[thm] = cands_in_proof

    # Add to the flat list of all candidate occurrences
    all_candidates_occurrences.extend(cands_in_proof)



Extracting candidates: 100%|██████████| 54475/54475 [00:07<00:00, 7360.76proof/s] 


In [ ]:
# Save the candidate occurrences counter to JSON, filtering to those found in premise_registry_unique.jsonl
import json
from collections import Counter

# Load all lemma names from premise_registry_unique.jsonl
allowed_lemmas_set = set()
with open("premise_registry_unique.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        try:
            entry = json.loads(line)
            allowed_lemmas_set.add(entry["full_name"])
        except Exception:
            continue

occurrence_counter = Counter(all_candidates_occurrences)

# Filter keys to only those that are in the allowed lemmas set
filtered_occurrence_counter = {lemma: count for lemma, count in occurrence_counter.items() if lemma in allowed_lemmas_set}
# Also store the 'filtered out' (not in allowed_lemmas_set)
filtered_out_occurrence_counter = {lemma: count for lemma, count in occurrence_counter.items() if lemma not in allowed_lemmas_set}

# Save filtered counts
with open("lemma_candidate_counter.json", "w", encoding="utf-8") as f:
    json.dump(filtered_occurrence_counter, f, indent=2, ensure_ascii=False)

# Save filtered out counts
with open("lemma_candidate_counter_filtered_out.json", "w", encoding="utf-8") as f:
    json.dump(filtered_out_occurrence_counter, f, indent=2, ensure_ascii=False)

percent_filtered = (len(filtered_occurrence_counter) / max(1, len(occurrence_counter))) * 100
percent_filtered_out = (len(filtered_out_occurrence_counter) / max(1, len(occurrence_counter))) * 100
print(f"Filtered %: {percent_filtered:.2f}% ({len(filtered_occurrence_counter)} / {len(occurrence_counter)})")
print(f"Filtered out %: {percent_filtered_out:.2f}% ({len(filtered_out_occurrence_counter)} / {len(occurrence_counter)})")
print(f"Saved filtered candidate occurrences to lemma_candidate_counter.json")
print(f"Saved filtered out candidate occurrences to lemma_candidate_counter_filtered_out.json")

Filtered %: 1.88% (1592 / 84627)
Filtered out %: 98.12% (83035 / 84627)
Saved filtered candidate occurrences to lemma_candidate_counter.json
Saved filtered out candidate occurrences to lemma_candidate_counter_filtered_out.json


In [ ]:
filtered_out_occurrence_counter

{'mk_uLift': 4,
 'x.coe_prop': 2,
 'lift_mk_le_lift_mk_mul_of_lift_mk_preimage_le': 1,
 'le_aleph0_iff_set_countable': 5,
 'lift_le_aleph0': 3,
 'MapsTo': 13,
 'Subtype.coe_injective.injOn': 9,
 'f.rootSet': 1,
 'f.rootSet_finite': 1,
 'this.countable_of_injOn': 1,
 'mem_rootSet': 1,
 'mem_rootSet.2': 3,
 'lift_id': 29,
 'cardinal_mk_lift_le_mul': 1,
 'cardinal_mk_lift_le_max': 1,
 'h': 6423,
 'only': 14447,
 'fst_inl': 7,
 'lift_apply_apply': 2,
 'snd_inl': 6,
 'zero_mul': 458,
 'inl_mul_eq_smul': 1,
 'lift_apply_inl': 4,
 'map_mul': 226,
 'mul_inl_eq_op_smul': 1,
 'fst_eps': 1,
 'map_one': 123,
 'snd_eps': 1,
 'isPrimePow_def': 3,
 "not_and'": 6,
 'pow_eq_zero': 10,
 'Nat.prime_iff': 6,
 'isPrimePow_nat_iff': 7,
 'Nat.lt_pow_self': 5,
 'hp.one_lt': 12,
 'p': 229,
 'hp.one_lt.le': 2,
 'Nat.dvd_prime_pow': 7,
 'not_true_eq_false': 26,
 'Finset.disjoint_left': 10,
 'Finset.mem_filter': 54,
 'Nat.mem_divisors': 7,
 'Nat.eq_one_of_dvd_coprimes': 2,
 'hn.ne_one': 1,
 'neZero_iff': 2,
 'mkS

## using corpus we already find more matched lemmas

In [ ]:
# Save the candidate occurrences counter to JSON, filtering to those found in corpus.jsonl
import json
from collections import Counter

# Load all lemma names from corpus.jsonl
allowed_lemmas_set = set()
with open("corpus.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        try:
            file_entry = json.loads(line)
            premises = file_entry.get("premises", [])
            for prem in premises:
                full_name = prem.get("full_name", "")
                if full_name:
                    allowed_lemmas_set.add(full_name)
        except Exception:
            continue

occurrence_counter = Counter(all_candidates_occurrences)

# Filter keys to only those that are in the allowed lemmas set
filtered_occurrence_counter = {lemma: count for lemma, count in occurrence_counter.items() if lemma in allowed_lemmas_set}
# Also store the 'filtered out' (not in allowed_lemmas_set)
filtered_out_occurrence_counter = {lemma: count for lemma, count in occurrence_counter.items() if lemma not in allowed_lemmas_set}

# Save filtered counts
with open("lemma_candidate_counter.json", "w", encoding="utf-8") as f:
    json.dump(filtered_occurrence_counter, f, indent=2, ensure_ascii=False)

# Save filtered out counts
with open("lemma_candidate_counter_filtered_out.json", "w", encoding="utf-8") as f:
    json.dump(filtered_out_occurrence_counter, f, indent=2, ensure_ascii=False)

percent_filtered = (len(filtered_occurrence_counter) / max(1, len(occurrence_counter))) * 100
percent_filtered_out = (len(filtered_out_occurrence_counter) / max(1, len(occurrence_counter))) * 100
print(f"Filtered %: {percent_filtered:.2f}% ({len(filtered_occurrence_counter)} / {len(occurrence_counter)})")
print(f"Filtered out %: {percent_filtered_out:.2f}% ({len(filtered_out_occurrence_counter)} / {len(occurrence_counter)})")
print(f"Saved filtered candidate occurrences to lemma_candidate_counter.json")
print(f"Saved filtered out candidate occurrences to lemma_candidate_counter_filtered_out.json")

Filtered %: 27.49% (23261 / 84627)
Filtered out %: 72.51% (61366 / 84627)
Saved filtered candidate occurrences to lemma_candidate_counter.json
Saved filtered out candidate occurrences to lemma_candidate_counter_filtered_out.json


In [ ]:
filtered_out_occurrence_counter


{'mk_uLift': 4,
 'x.coe_prop': 2,
 'lift_mk_le_lift_mk_mul_of_lift_mk_preimage_le': 1,
 'le_aleph0_iff_set_countable': 5,
 'lift_le_aleph0': 3,
 'MapsTo': 13,
 'Subtype.coe_injective.injOn': 9,
 'f.rootSet': 1,
 'f.rootSet_finite': 1,
 'this.countable_of_injOn': 1,
 'mem_rootSet': 1,
 'mem_rootSet.2': 3,
 'lift_id': 29,
 'cardinal_mk_lift_le_mul': 1,
 'cardinal_mk_lift_le_max': 1,
 'h': 6423,
 'only': 14447,
 'add_zero': 523,
 'fst_inl': 7,
 'lift_apply_apply': 2,
 'map_zero': 245,
 'snd_inl': 6,
 'zero_mul': 458,
 'inl_mul_eq_smul': 1,
 'lift_apply_inl': 4,
 'mul_inl_eq_op_smul': 1,
 'fst_eps': 1,
 'snd_eps': 1,
 'zero_add': 530,
 'hp.one_lt': 12,
 'p': 229,
 'hp.one_lt.le': 2,
 'hn.ne_one': 1,
 'mkSol': 3,
 'n.is_lt': 1,
 'mod_cast': 202,
 'E.order': 3,
 'add_comm': 830,
 'not_lt.mp': 25,
 'add_lt_add_right': 5,
 'k.is_lt': 2,
 'eq_mk_of_is_sol_of_eq_init': 1,
 'Iff.intro': 24,
 'E.solSpace': 2,
 "u'.val": 1,
 "v'.val": 1,
 "h'": 387,
 'E.toInit.toEquiv.apply_eq_iff_eq': 1,
 'mem_ran

## running proof with tactics and resolving at proof time? why are we doing this

In [ ]:

# Run proof tactics with the new signature
result = run_proof_tactics(
    i=0,  # proof index
    data=data,
    theorem_registry=theorem_registry,  # Use theorem_registry instead of edges
    premise_registry=premise_registry,  # Use premise_registry instead of edges
    printing=True,
    print_states=False,
    show_new_lemmas_per_step=True
)

# Access results
print(f"Success: {result['success']}")
print(f"Theorem: {result['theorem_full_name']}")
print(f"Resolved lemmas: {len(result['resolved_best'])}")
print(f"Unresolved: {len(result['unresolved'])}")

Running tactics for proof index: 0
Theorem: Theorem(repo=LeanGitRepo(url='https://github.com/leanprover-community/mathlib4', commit='29dcec074de168ac2bf835a77ef68bbe069194c5'), file_path=WindowsPath('Mathlib/Algebra/AlgebraicCard.lean'), full_name='Algebraic.cardinal_mk_lift_le_mul')


proof[0] tactics:  14%|█▍        | 1/7 [00:00<00:00,  6.80tac/s]


>>> Applying tactic block #0:
  rw [← mk_uLift, ← mk_uLift]
>>> New lemma candidates observed:
    - mk_uLift  (unresolved, tried: ['Cardinal.mk_uLift', 'mk_uLift'])

>>> Applying tactic block #1:
  choose g hg₁ hg₂ using fun x : { x : A | IsAlgebraic R x } => x.coe_prop


proof[0] tactics:  29%|██▊       | 2/7 [00:00<00:00,  5.11tac/s]


>>> Applying tactic block #2:
  refine lift_mk_le_lift_mk_mul_of_lift_mk_preimage_le g fun f => ?_
>>> New lemma candidates observed:
    - lift_mk_le_lift_mk_mul_of_lift_mk_preimage_le  (unresolved, tried: ['lift_mk_le_lift_mk_mul_of_lift_mk_preimage_le'])


...


>>> Applying tactic block #4:
  suffices MapsTo (↑) (g ⁻¹' {f}) (f.rootSet A) from
    this.countable_of_injOn Subtype.coe_injective.injOn (f.rootSet_finite A).countable
>>> New lemma candidates observed:
    - MapsTo  (unresolved, tried: ['Set.MapsTo', 'MapsTo'])
    - Subtype.coe_injective.injOn  (unresolved, tried: ['Function.Injective.injOn', 'Subtype.coe_injective.injOn'])
    - f.rootSet  (unresolved, tried: ['Polynomial.rootSet', 'f.rootSet'])
    - f.rootSet_finite  (unresolved, tried: ['Polynomial.rootSet_finite', 'f.rootSet_finite'])
    - this.countable_of_injOn  (unresolved, tried: ['Set.MapsTo.countable_of_injOn'])


proof[0] tactics:  71%|███████▏  | 5/7 [00:01<00:00,  4.40tac/s]


>>> Applying tactic block #5:
  rintro x (rfl : g x = f)


proof[0] tactics: 100%|██████████| 7/7 [00:01<00:00,  5.07tac/s]


>>> Applying tactic block #6:
  exact mem_rootSet.2 ⟨hg₁ x, hg₂ x⟩
>>> New lemma candidates observed:
    - mem_rootSet  (unresolved, tried: ['Polynomial.mem_rootSet', 'mem_rootSet'])
    - mem_rootSet.2  (unresolved, tried: ['Polynomial.mem_rootSet', 'mem_rootSet'])

Summary for proof index 0
  success: True
  theorem: Algebraic.cardinal_mk_lift_le_mul
  file:    Mathlib\Algebra\AlgebraicCard.lean
  tactic blocks: 7
  unique candidates: 12
  resolved: 0
  unresolved: 11
  ambiguous: 0
  unresolved (sample): ['MapsTo', 'Subtype.coe_injective.injOn', 'f.rootSet', 'f.rootSet_finite', 'le_aleph0_iff_set_countable', 'lift_le_aleph0', 'lift_mk_le_lift_mk_mul_of_lift_mk_preimage_le', 'mem_rootSet', 'mem_rootSet.2', 'mk_uLift', 'this.countable_of_injOn']
  normalized names tried (sample):
    MapsTo -> ['Set.MapsTo', 'MapsTo']
    Subtype.coe_injective.injOn -> ['Function.Injective.injOn', 'Subtype.coe_injective.injOn']
    f.rootSet -> ['Polynomial.rootSet', 'f.rootSet']
    f.rootSet_finit

In [ ]:

# Run proof tactics for a specific proof
result = run_proof_tactics(
    i=0,  # proof index
    data=data,
    edges=edges,
    printing=True,
    print_states=False,
    show_new_lemmas_per_step=True
)

# Access results
print(f"Success: {result['success']}")
print(f"Theorem: {result['theorem_full_name']}")
print(f"Resolved lemmas: {len(result['resolved_best'])}")
print(f"Unresolved: {len(result['unresolved'])}")

Running tactics for proof index: 0
Theorem: Theorem(repo=LeanGitRepo(url='https://github.com/leanprover-community/mathlib4', commit='29dcec074de168ac2bf835a77ef68bbe069194c5'), file_path=WindowsPath('Mathlib/Algebra/AlgebraicCard.lean'), full_name='Algebraic.cardinal_mk_lift_le_mul')


proof[0] tactics:  14%|█▍        | 1/7 [00:00<00:00,  6.27tac/s]


>>> Applying tactic block #0:
  rw [← mk_uLift, ← mk_uLift]
>>> New lemma candidates observed:
    - mk_uLift  (unresolved)

>>> Applying tactic block #1:
  choose g hg₁ hg₂ using fun x : { x : A | IsAlgebraic R x } => x.coe_prop


proof[0] tactics:  29%|██▊       | 2/7 [00:00<00:01,  4.11tac/s]


>>> Applying tactic block #2:
  refine lift_mk_le_lift_mk_mul_of_lift_mk_preimage_le g fun f => ?_
>>> New lemma candidates observed:
    + lift_mk_le_lift_mk_mul_of_lift_mk_preimage_le  ->  Cardinal.lift_mk_le_lift_mk_mul_of_lift_mk_preimage_le   (Mathlib\SetTheory\Cardinal\Basic.lean)


...


>>> Applying tactic block #4:
  suffices MapsTo (↑) (g ⁻¹' {f}) (f.rootSet A) from
    this.countable_of_injOn Subtype.coe_injective.injOn (f.rootSet_finite A).countable
>>> New lemma candidates observed:
    + Subtype.coe_injective.injOn  ->  CompleteLattice.Independent.injOn   (Mathlib\Order\SupIndep.lean)
    - MapsTo  (unresolved)
    - f.rootSet  (unresolved)
    - f.rootSet_finite  (unresolved)
    - this.countable_of_injOn  (unresolved)


proof[0] tactics:  71%|███████▏  | 5/7 [00:01<00:00,  3.99tac/s]


>>> Applying tactic block #5:
  rintro x (rfl : g x = f)


proof[0] tactics: 100%|██████████| 7/7 [00:01<00:00,  4.68tac/s]


>>> Applying tactic block #6:
  exact mem_rootSet.2 ⟨hg₁ x, hg₂ x⟩
>>> New lemma candidates observed:
    + mem_rootSet  ->  Polynomial.mem_rootSet   (Mathlib\Algebra\Polynomial\Roots.lean)
    + mem_rootSet.2  ->  Polynomial.mem_rootSet   (Mathlib\Algebra\Polynomial\Roots.lean)

Summary for proof index 0
  success: True
  theorem: Algebraic.cardinal_mk_lift_le_mul
  file:    Mathlib\Algebra\AlgebraicCard.lean
  tactic blocks: 7
  unique candidates: 12
  resolved: 5
  unresolved: 6
  ambiguous: 0
  unresolved (sample): ['MapsTo', 'f.rootSet', 'f.rootSet_finite', 'le_aleph0_iff_set_countable', 'mk_uLift', 'this.countable_of_injOn']
Success: True
Theorem: Algebraic.cardinal_mk_lift_le_mul
Resolved lemmas: 5
Unresolved: 6
